In [0]:
# Configuration & Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import time

CATALOG = "fahrlytics"
BRONZE  = f"{CATALOG}.bronze"
SILVER  = f"{CATALOG}.silver"

spark.sql(f"USE CATALOG {CATALOG}")

print(f"Catalog : {CATALOG}")
print(f"Source  : {BRONZE}.payments (audited against {SILVER}.trips)")
print(f"Target  : {SILVER}.payments")

In [0]:
# Determine Load Strategy (Full vs. Incremental)
print("Evaluating load strategy...")

table_exists = spark.catalog.tableExists(f"{SILVER}.payments")
is_empty = not table_exists or spark.table(f"{SILVER}.payments").count() == 0

if is_empty:
    LOAD_TYPE = "full"
    df_bronze = spark.table(f"{BRONZE}.payments")
    print("Initiating FULL load for payments.")
else:
    LOAD_TYPE = "incremental"
    last_run = spark.sql(f"SELECT MAX(_processed_at) FROM {SILVER}.payments").collect()[0][0]
    df_bronze = spark.table(f"{BRONZE}.payments").filter(F.col("_ingested_at") > last_run)
    print(f"Initiating INCREMENTAL load for payments ingested after: {last_run}")

input_count = df_bronze.count()
print(f"Incoming payment rows: {input_count:,}")

if input_count == 0:
    print("No new payment rows to process. Exiting.")
    dbutils.notebook.exit("No new payments")

In [0]:
# Deduplication & Type Casting
print("Deduplicating and standardizing payments...")

df_deduped = df_bronze.dropDuplicates(["payment_id"])
dedup_removed = input_count - df_deduped.count()
print(f"Deduplicated count: {df_deduped.count():,} (Dropped {dedup_removed:,} duplicate keys)")

df_standardized = (
    df_deduped
    .withColumn("payment_timestamp", F.to_timestamp("payment_timestamp"))
    .withColumn("amount", F.round(F.col("amount").cast("double"), 2))
    .withColumn("currency", F.coalesce(F.col("currency"), F.lit("EUR")))
    .withColumn("payment_status", F.lower(F.trim(F.col("payment_status"))))
    .withColumn("payment_method", F.trim(F.col("payment_method")))
    .withColumn("card_hash", F.when(F.col("card_hash") == "", None).otherwise(F.col("card_hash")))
)

In [0]:
#  Referential Integrity Join (Orphan & Fare Mismatch Auditing)
print("Auditing against silver.trips for orphan transactions and billing mismatches...")

# Project only necessary columns from silver.trips to minimize shuffle memory footprint
df_trips_ref = spark.table(f"{SILVER}.trips").select(
    F.col("trip_id").alias("ref_trip_id"),
    F.col("customer_id").alias("trip_customer_id"),
    F.col("city").alias("trip_city"),
    F.col("fare_amount").alias("expected_fare_amount"),
    F.col("trip_status").alias("ref_trip_status")
)

df_joined = df_standardized.join(
    df_trips_ref,
    df_standardized.trip_id == df_trips_ref.ref_trip_id,
    how="left"
)

df_audited = (
    df_joined
    # 1. Orphan payment flag: Payment exists but no corresponding trip record exists
    .withColumn("is_orphan_payment", F.col("ref_trip_id").isNull())
    
    # 2. Customer mismatch check: If trip exists, verify payment belongs to booking customer
    .withColumn("is_customer_mismatched", 
        F.when(F.col("ref_trip_id").isNotNull() & (F.col("customer_id") != F.col("trip_customer_id")), True)
        .otherwise(False)
    )
    
    # 3. Fare mismatch check: Difference greater than 1.00 EUR tolerance
    .withColumn("fare_difference", 
        F.when(F.col("ref_trip_id").isNotNull(), F.round(F.abs(F.col("amount") - F.col("expected_fare_amount")), 2))
        .otherwise(None)
    )
    .withColumn("is_fare_mismatch", 
        F.when(F.col("fare_difference") > 1.00, True)
        .otherwise(False)
    )
    
    # Inherit canonical city from trip; fallback to "Unknown" for orphan payments
    .withColumn("city", F.coalesce(F.col("trip_city"), F.lit("Unknown")))
    .drop("ref_trip_id", "trip_customer_id", "trip_city", "expected_fare_amount", "ref_trip_status")
)

In [0]:
# Credit Card Fraud Ring Detection
print("Auditing multi-account credit card reuse...")

# Calculate distinct customer count per card hash across all credit card rows
card_window = Window.partitionBy("card_hash")

df_fraud_flagged = (
    df_audited
    .withColumn("card_user_count", 
        F.when(F.col("card_hash").isNotNull(), F.size(F.collect_set("customer_id").over(card_window)))
        .otherwise(1)
    )
    # Flag cards linked to 3 or more distinct accounts
    .withColumn("is_shared_card_fraud", 
        F.when(F.col("card_user_count") >= 3, True).otherwise(F.col("is_fraud_card"))
    )
    .drop("card_user_count")
    .withColumn("_processed_at", F.current_timestamp())
)

orphan_total = df_fraud_flagged.filter(F.col("is_orphan_payment") == True).count()
mismatch_total = df_fraud_flagged.filter(F.col("is_fare_mismatch") == True).count()
fraud_card_total = df_fraud_flagged.filter(F.col("is_shared_card_fraud") == True).count()

print(f"  Orphan payments flagged    : {orphan_total:,}")
print(f"  Fare mismatches flagged    : {mismatch_total:,}")
print(f"  Fraudulent cards detected  : {fraud_card_total:,}")

In [0]:
# Create Silver Payments Delta Table (if not exists)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER}.payments (
        payment_id              STRING,
        trip_id                 STRING,
        customer_id             STRING,
        city                    STRING,
        amount                  DOUBLE,
        currency                STRING,
        payment_method          STRING,
        payment_timestamp       TIMESTAMP,
        payment_status          STRING,
        card_hash               STRING,
        is_fraud_card           BOOLEAN,
        is_orphan_payment       BOOLEAN,
        is_customer_mismatched  BOOLEAN,
        fare_difference         DOUBLE,
        is_fare_mismatch        BOOLEAN,
        is_shared_card_fraud    BOOLEAN,
        _ingested_at            TIMESTAMP,
        _processed_at           TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (payment_status)
    COMMENT 'Silver reconciled payments ledger with fraud and audit flags'
""")

In [0]:
# Write or Merge into Silver Payments
print(f"Writing to {SILVER}.payments (Mode: {LOAD_TYPE})...")
t0 = time.time()

if LOAD_TYPE == "full":
    (
        df_fraud_flagged.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("payment_status")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SILVER}.payments")
    )
    print(f"Full Silver payments load written: {df_fraud_flagged.count():,} rows")
else:
    target_payments = DeltaTable.forName(spark, f"{SILVER}.payments")
    (
        target_payments.alias("target")
        .merge(
            df_fraud_flagged.alias("source"),
            "target.payment_status = source.payment_status AND target.payment_id = source.payment_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Incremental payment merge finished.")

print(f"Write duration: {round(time.time() - t0, 2)}s")

# File compaction
print("Running Delta OPTIMIZE...")
spark.sql(f"OPTIMIZE {SILVER}.payments")
print("Delta OPTIMIZE completed.")

In [0]:
# Verification & Audit Breakdown
print("=== FAHRLYTICS SILVER PAYMENTS VERIFICATION ===\n")

df_verify = spark.table(f"{SILVER}.payments")
total_rows = df_verify.count()
print(f"Total payments in Silver: {total_rows:,}\n")

print("Payment Status Distribution:")
df_verify.groupBy("payment_status").count().show()

print("Payment Method Breakdown:")
df_verify.groupBy("payment_method").count().orderBy("count", ascending=False).show()

print("Audit & Anomaly Summary:")
df_verify.selectExpr(
    "count(case when is_orphan_payment then 1 end) as orphan_payments",
    "count(case when is_fare_mismatch then 1 end) as fare_mismatches",
    "count(case when is_customer_mismatched then 1 end) as customer_mismatches",
    "count(case when is_shared_card_fraud then 1 end) as confirmed_fraud_cards"
).show()